# 第54章 分面图（FacetGrid）

使用FacetGrid、catplot和relplot把类别映射为可比较的小图。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

一个图过于拥挤，需要按行列分组重复相同图形。

## 数据结构

长表，包含X、Y以及一至两个分面分类变量。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 col_wrap=3 改为 col_wrap=2，观察分面换行对布局的影响
2. 修改 sharex=True 为 sharex=False，对比共享与独立X轴对子图比较的作用
3. 调整 aspect 参数（如 0.8 或 1.2），说明子图宽高比对可读性的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
grid = sns.relplot(data=marketing, x="visits", y="sales", col="channel", col_wrap=3, hue="channel", height=3.3, aspect=1, palette="colorblind", legend=False)
grid.set_axis_labels("访问量", "销售额")
grid.set_titles("{col_name}")
grid.fig.suptitle("分渠道访问量与销售额", y=1.04)
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
grid = sns.catplot(data=orders, x="category", y="order_value", col="region", kind="box", hue="category", palette="Set2", legend=False, height=3.5, aspect=0.9)
grid.set_axis_labels("品类", "客单价（元）")
grid.set_titles("{col_name}")
grid.fig.suptitle("分区域品类客单价", y=1.04)
plt.show()


## 3. 参数说明

- row/col：分面
- col_wrap：换行
- sharex/sharey：共享轴
- height/aspect：尺寸


## 4. 结果解读

在共享坐标下比较模式、斜率和分布；同时检查每个面板样本量。


## 常见误区

- 面板过多
- 坐标不共享却直接比较高低
- 分面和hue重复编码同一变量


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
facet = sns.FacetGrid(daily, row="region", height=2.2, aspect=3, sharex=True, sharey=True, margin_titles=True)
facet.map_dataframe(sns.lineplot, x="date", y="sales", marker="o", errorbar=None, color="#1a73e8")
facet.set_axis_labels("日期", "销售额")
facet.set_titles(row_template="{row_name}")
facet.fig.subplots_adjust(top=0.9)
facet.fig.suptitle("区域每日销售趋势")
plt.show()


## 本章小结

使用FacetGrid、catplot和relplot把类别映射为可比较的小图。


### 你已经掌握

- 判断分面图（FacetGrid）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 一个图过于拥挤，需要按行列分组重复相同图形。 |
| 数据结构 | 长表，包含X、Y以及一至两个分面分类变量。 |
| 结果解读 | 在共享坐标下比较模式、斜率和分布；同时检查每个面板样本量。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `row/col` | 分面 |
| `col_wrap` | 换行 |
| `sharex/sharey` | 共享轴 |
| `height/aspect` | 尺寸 |


### 需要注意

- 面板过多
- 坐标不共享却直接比较高低
- 分面和hue重复编码同一变量


### 完成检查

- [ ] 能判断什么问题适合使用分面图（FacetGrid）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
